## Menyiapkan DataSet

In [1]:
import numpy as np
import pandas as pd

np.random.seed(212)
n = 3000
menu_list = ["Kopi Susu", "Americano", "Cappuccino", "Roti Bakar", "Kentang Goreng", "Es Teh"]
cabang_list = ["Magelang", "Yogyakarta", "Semarang"]
waktu_list = ["Pagi", "Siang", "Sore", "Malam"]

data_tugas12 = pd.DataFrame({
    "nama_menu": np.random.choice(menu_list, size=n),
    "cabang": np.random.choice(cabang_list, size=n),
    "waktu_pesan": np.random.choice(waktu_list, size=n),
    "qty": np.random.randint(1, 6, size=n),
    "waktu_tunggu_menit": np.random.randint(2, 25, size=n),
})

# Simulasi: waktu tunggu lebih lama cenderung menurunkan peluang ulasan positif
prob_positif = 1 / (1 + np.exp((data_tugas12["waktu_tunggu_menit"] - 12) / 4))
data_tugas12["ulasan_positif"] = (np.random.rand(n) < prob_positif).astype(int)

data_tugas12.to_csv("tugas12_ulasan.csv", index=False)
print(f"Dataset berhasil dibuat: {len(data_tugas12)} baris")
print("Distribusi label:")
print(data_tugas12["ulasan_positif"].value_counts())

Dataset berhasil dibuat: 3000 baris
Distribusi label:
ulasan_positif
0    1610
1    1390
Name: count, dtype: int64


## Membuat SparkSession

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = SparkSession.builder.appName("Tugas12-KopiNusantara").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

df_tugas = spark.read.csv("tugas12_ulasan.csv", header=True, inferSchema=True)
df_tugas.printSchema()
df_tugas.show(5)
print("Total baris:", df_tugas.count())

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/22 13:12:50 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
                                                                                

root
 |-- nama_menu: string (nullable = true)
 |-- cabang: string (nullable = true)
 |-- waktu_pesan: string (nullable = true)
 |-- qty: integer (nullable = true)
 |-- waktu_tunggu_menit: integer (nullable = true)
 |-- ulasan_positif: integer (nullable = true)

+--------------+----------+-----------+---+------------------+--------------+
|     nama_menu|    cabang|waktu_pesan|qty|waktu_tunggu_menit|ulasan_positif|
+--------------+----------+-----------+---+------------------+--------------+
|     Kopi Susu|Yogyakarta|       Sore|  5|                 5|             1|
|        Es Teh|Yogyakarta|       Pagi|  1|                21|             0|
|    Cappuccino|Yogyakarta|       Sore|  5|                15|             0|
|Kentang Goreng|  Semarang|      Malam|  2|                 5|             1|
|     Americano|Yogyakarta|      Malam|  2|                10|             0|
+--------------+----------+-----------+---+------------------+--------------+
only showing top 5 rows

Total baris

___
### A. Membangun ML Pipeline

In [3]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator

idx_menu = StringIndexer(inputCol="nama_menu", outputCol="menu_idx")
idx_cabang = StringIndexer(inputCol="cabang", outputCol="cabang_idx")
idx_waktu = StringIndexer(inputCol="waktu_pesan", outputCol="waktu_idx")

ohe_t = OneHotEncoder(
    inputCols=["menu_idx", "cabang_idx", "waktu_idx"],
    outputCols=["menu_vec", "cabang_vec", "waktu_vec"]
)

assembler_t = VectorAssembler(
    inputCols=["menu_vec", "cabang_vec", "waktu_vec", "qty", "waktu_tunggu_menit"],
    outputCol="features"
)

lr_t = LogisticRegression(featuresCol="features", labelCol="ulasan_positif")

pipeline_lr = Pipeline(stages=[idx_menu, idx_cabang, idx_waktu, ohe_t, assembler_t, lr_t])
print("Pipeline berhasil dibuat.")

Pipeline berhasil dibuat.


### B. Pelatihan dan Evaluasi

In [4]:
train_t, test_t = df_tugas.randomSplit([0.8, 0.2], seed=77)

model_lr = pipeline_lr.fit(train_t)
pred_lr = model_lr.transform(test_t)

ev_acc_t = MulticlassClassificationEvaluator(labelCol="ulasan_positif", predictionCol="prediction", metricName="accuracy")
ev_f1_t = MulticlassClassificationEvaluator(labelCol="ulasan_positif", predictionCol="prediction", metricName="f1")
ev_auc_t = BinaryClassificationEvaluator(labelCol="ulasan_positif", metricName="areaUnderROC")

acc_lr = ev_acc_t.evaluate(pred_lr)
f1_lr = ev_f1_t.evaluate(pred_lr)
auc_lr = ev_auc_t.evaluate(pred_lr)

print("=== Logistic Regression ===")
print(f"Accuracy : {acc_lr:.4f}")
print(f"F1-score : {f1_lr:.4f}")
print(f"AUC-ROC  : {auc_lr:.4f}")

print("\nConfusion Matrix:")
pred_lr.groupBy("ulasan_positif", "prediction").count().orderBy("ulasan_positif", "prediction").show()

=== Logistic Regression ===
Accuracy : 0.7929
F1-score : 0.7926
AUC-ROC  : 0.8650

Confusion Matrix:
+--------------+----------+-----+
|ulasan_positif|prediction|count|
+--------------+----------+-----+
|             0|       0.0|  257|
|             0|       1.0|   57|
|             1|       0.0|   65|
|             1|       1.0|  210|
+--------------+----------+-----+



### C. Model Pembanding Random Forest

In [5]:
rf_t = RandomForestClassifier(featuresCol="features", labelCol="ulasan_positif", numTrees=30)

pipeline_rf = Pipeline(stages=[idx_menu, idx_cabang, idx_waktu, ohe_t, assembler_t, rf_t])

model_rf = pipeline_rf.fit(train_t)
pred_rf = model_rf.transform(test_t)

acc_rf = ev_acc_t.evaluate(pred_rf)
f1_rf = ev_f1_t.evaluate(pred_rf)
auc_rf = ev_auc_t.evaluate(pred_rf)

print("=== Perbandingan Model ===")
print(f"Logistic Regression -> Accuracy: {acc_lr:.4f} | F1: {f1_lr:.4f} | AUC: {auc_lr:.4f}")
print(f"Random Forest        -> Accuracy: {acc_rf:.4f} | F1: {f1_rf:.4f} | AUC: {auc_rf:.4f}")

=== Perbandingan Model ===
Logistic Regression -> Accuracy: 0.7929 | F1: 0.7926 | AUC: 0.8650
Random Forest        -> Accuracy: 0.7980 | F1: 0.7974 | AUC: 0.8426


### D. Analisis Kesalahan Model

In [6]:
if auc_rf > auc_lr:
    pred_terbaik, nama_terbaik = pred_rf, "Random Forest"
else:
    pred_terbaik, nama_terbaik = pred_lr, "Logistic Regression"

print("Model terbaik:", nama_terbaik)

salah = pred_terbaik.filter(col("ulasan_positif") != col("prediction"))

salah.select("nama_menu", "cabang", "waktu_pesan", "qty",
             "waktu_tunggu_menit", "ulasan_positif", "prediction").show(10)

Model terbaik: Logistic Regression
+---------+--------+-----------+---+------------------+--------------+----------+
|nama_menu|  cabang|waktu_pesan|qty|waktu_tunggu_menit|ulasan_positif|prediction|
+---------+--------+-----------+---+------------------+--------------+----------+
|Americano|Magelang|      Malam|  1|                15|             1|       0.0|
|Americano|Magelang|      Siang|  2|                22|             1|       0.0|
|Americano|Magelang|       Sore|  1|                 7|             0|       1.0|
|Americano|Magelang|       Sore|  2|                11|             0|       1.0|
|Americano|Magelang|       Sore|  3|                 5|             0|       1.0|
|Americano|Magelang|       Sore|  3|                13|             1|       0.0|
|Americano|Magelang|       Sore|  5|                 6|             0|       1.0|
|Americano|Magelang|       Sore|  5|                15|             1|       0.0|
|Americano|Semarang|      Malam|  1|                 2|        

In [7]:
salah.groupBy("waktu_tunggu_menit").count().orderBy("waktu_tunggu_menit").show(25)

+------------------+-----+
|waktu_tunggu_menit|count|
+------------------+-----+
|                 2|    1|
|                 3|    3|
|                 4|    2|
|                 5|    2|
|                 6|    6|
|                 7|    5|
|                 8|    7|
|                 9|    6|
|                10|    8|
|                11|   11|
|                12|   10|
|                13|   10|
|                14|   13|
|                15|    6|
|                16|    8|
|                17|    6|
|                18|    2|
|                19|    4|
|                20|    2|
|                21|    1|
|                22|    4|
|                23|    1|
|                24|    4|
+------------------+-----+



### E. Kesimpulan Bisnis
Waktu tunggu menjadi faktor utama yang memengaruhi ulasan pelanggan. Kesalahan prediksi paling banyak terjadi pada rentang 11–14 menit karena peluang ulasan positif dan negatif hampir seimbang. Berdasarkan hasil tersebut, manajemen Kopi Nusantara disarankan menetapkan target penyajian maksimal 10 menit, menambah staf saat jam ramai, serta memberikan kompensasi bagi pelanggan yang menunggu lebih dari 15 menit. Model Logistic Regression dengan AUC 0,8650 dapat digunakan sebagai peringatan dini untuk mendeteksi pesanan berisiko mendapat ulasan negatif.

In [9]:
# Jika semua sudah selesai, jangan lupa tutup SparkSession :
spark.stop()
print("SparkSession ditutup.")

SparkSession ditutup.
